In [ ]:
import os
from joblib import load
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, matthews_corrcoef, roc_auc_score
import time

In [ ]:
def metrics(predict, y, dataset, sampling, model_name, representation, predict_proba=None, fit_time=None, score_time=None):
    acc = accuracy_score(y, predict)
    recall = recall_score(y, predict, average='weighted')
    precision = precision_score(y, predict, average='weighted')
    f1 = f1_score(y, predict, average='weighted')
    mcc = matthews_corrcoef(y, predict)
    roc_auc = None
    if predict_proba is not None:
        try:
            roc_auc = roc_auc_score(y, predict_proba, multi_class='ovr', average='weighted')
        except Exception as e:
            print(f"[{model_name}] Error al calcular ROC AUC: {e}")
    return pd.DataFrame([[
        dataset, model_name, sampling, representation,
        acc, recall, precision, f1, roc_auc, mcc,
        fit_time, score_time
    ]], columns=[
        "dataset", "model", "sampling", "representation",
        "accuracy", "recall", "precision", "f1",
        "roc_auc_ovr", "mcc", "fit_time", "score_time"
    ])

In [ ]:
model_dir = "../../data/data_grip/best/"
test_dir = "../numerical_rep/indep/"
resultados = []

In [ ]:
carpeta = "best/"

if os.path.exists(carpeta):
    print("Carpeta encontrada:", carpeta)

Carpeta encontrada: best/


In [ ]:
try:
    archivos = os.listdir(carpeta)
    print("📁 Archivos en la carpeta:")
    for archivo in archivos:
        print(" -", archivo)
except Exception as e:
    print("❌ No se pudo acceder a la carpeta:", e)

📁 Archivos en la carpeta:
 - embedding_antiviral_homology_90_protT5_AdaBoost_Grid_42_Original_best.joblib
 - embedding_antiviral_homology_90_protT5_AdaBoost_Grid_42_Over_best.joblib
 - embedding_antiviral_homology_90_protT5_AdaBoost_Grid_42_Under_best.joblib
 - embedding_antiviral_homology_90_protT5_GradientBoosting_Grid_42_Original_best.joblib
 - embedding_antiviral_homology_90_protT5_GradientBoosting_Grid_42_Over_best.joblib
 - embedding_antiviral_homology_90_protT5_GradientBoosting_Grid_42_Under_best.joblib
 - embedding_antiviral_homology_90_protT5_KNN_Grid_42_Original_best.joblib
 - embedding_antiviral_homology_90_protT5_KNN_Grid_42_Over_best.joblib
 - embedding_antiviral_homology_90_protT5_KNN_Grid_42_Under_best.joblib
 - embedding_antiviral_homology_90_protT5_LGBM_Grid_42_Original_best.joblib
 - embedding_antiviral_homology_90_protT5_LGBM_Grid_42_Under_best.joblib
 - embedding_antiviral_homology_90_protT5_LogisticRegression_Grid_42_Original_best.joblib
 - embedding_antiviral_homo

In [ ]:
model_path = "best/embedding_antiviral_homology_90_protT5_AdaBoost_Grid_42_Original_best.joblib"

In [ ]:
print("Ruta absoluta:", os.path.abspath(model_path))
print("¿Existe el archivo?:", os.path.exists(model_path))

Ruta absoluta: c:\Users\hantr\OneDrive\Documentos\Half_life\data\data_grip\best\embedding_antiviral_homology_90_protT5_AdaBoost_Grid_42_Original_best.joblib
¿Existe el archivo?: True


In [ ]:
import joblib
print("Versión de joblib:", joblib.__version__)

Versión de joblib: 1.1.0


In [ ]:
model = load('best/embedding_antiviral_homology_90_protT5_RandomForest_Grid_42_Original_best.joblib')

KeyError: 2

In [ ]:
for model_file in os.listdir(model_dir):
    if not model_file.endswith('.joblib'):
        continue

    # Parsear el nombre del modelo
    parts = model_file.split('_')
    representation_type = parts[0]  # fft, embedding, pc
    representation_name = parts[4]  # p.ej. FASG760101 (para fft y pc)
    algorithm = parts[5]            # KNN, RandomForest, etc.
    sampling = parts[7]             # Original, Over, Under

    # --------------------
    # Buscar el archivo de test correspondiente
    # --------------------
    if representation_type in ['fft', 'pc']:
        test_file = f"{representation_type}_excluded_homology_{representation_name}.csv"
        test_path = os.path.join(test_dir, representation_type, test_file)
    else:
        # Embedding, onehot, frequency, etc.
        test_file = f"{representation_type}.csv"
        test_path = os.path.join(test_dir, test_file)

    if not os.path.exists(test_path):
        print(f"Test file {test_file} no encontrado para modelo {model_file}")
        continue
    
    # Cargar el testeo
    test_df = pd.read_csv(test_path)
    test_df=test_df.drop(columns=['experimental_characteristics'])
    X_test = test_df.drop(columns=['target'])
    y_test = test_df['target']

    # Cargar el modelo
    model_path = os.path.join(model_dir, model_file)
    model = joblib.load(model_path)

    # Ejecutar predicción
    start_time = time.time()
    predict = model.predict(X_test)
    score_time = time.time() - start_time

    try:
        predict_proba = model.predict_proba(X_test)
    except:
        predict_proba = None

    # Guardar métricas
    df_metrics = metrics(
        predict, y_test, dataset='Validation',
        sampling=sampling, model_name=algorithm,
        representation=f"{representation_type}_{representation_name}_{algorithm}_42_metrics",
        predict_proba=predict_proba, fit_time=None, score_time=score_time
    )

    resultados.append(df_metrics)


Test file embedding.csv no encontrado para modelo embedding_antiviral_homology_90_protT5_AdaBoost_Grid_42_Original_best.joblib
Test file embedding.csv no encontrado para modelo embedding_antiviral_homology_90_protT5_AdaBoost_Grid_42_Over_best.joblib
Test file embedding.csv no encontrado para modelo embedding_antiviral_homology_90_protT5_AdaBoost_Grid_42_Under_best.joblib
Test file embedding.csv no encontrado para modelo embedding_antiviral_homology_90_protT5_GradientBoosting_Grid_42_Original_best.joblib
Test file embedding.csv no encontrado para modelo embedding_antiviral_homology_90_protT5_GradientBoosting_Grid_42_Over_best.joblib
Test file embedding.csv no encontrado para modelo embedding_antiviral_homology_90_protT5_GradientBoosting_Grid_42_Under_best.joblib
Test file embedding.csv no encontrado para modelo embedding_antiviral_homology_90_protT5_KNN_Grid_42_Original_best.joblib
Test file embedding.csv no encontrado para modelo embedding_antiviral_homology_90_protT5_KNN_Grid_42_Over_

KeyError: 0